Setting the python path to import code from app/src.

The extrapath is configured at .vscode/settings.json to avoid IDE errors.

In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="pyproject.toml") -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} not found starting from {p}")


root_dir = find_project_root() / "app" / "src"
sys.path.append(str(root_dir))

Just trying to see how well OCR works on a sample image from a non-readable PDF

In [ ]:
from pathlib import Path
import fitz


doc_path = Path("../../data/policies/raw/15414618005202301.pdf")
dpi = 150

doc = fitz.open(doc_path)
page = doc[3]
zoom = dpi / 72
matrix = fitz.Matrix(zoom, zoom)
pix = page.get_pixmap(matrix=matrix)
doc.close()


In [ ]:
from PIL import Image
import io

image = Image.open(io.BytesIO(pix.tobytes("png")))
image.show()

pytesseract is the python wrapper for tesseract OCR

To use tesseract OCR you need to have tesseract installed on your system and the specific language data files for the language you want to recognize. In this case, for example, I am using the portuguese language.

Just for documentation purposes, in linux with apt:

```bash
sudo apt update
sudo apt install tesseract-ocr tesseract-ocr-por
``` 

and check the installation with:

```bash
tesseract --version
```

the language can be checked with:

```bash
tesseract --list-langs
```

In [ ]:
import pytesseract 

text = pytesseract.image_to_string(image, lang="por")

print(text)

Now, I'll try a simple cache implementation to avoid reprocessing the same image multiple times.

In [ ]:
import pytesseract
import fitz
from PIL import Image
import io

def rasterize_page(doc_path, page_num, dpi=300):
    doc = fitz.open(doc_path)
    page = doc[page_num]
    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix)
    doc.close()
    return pix

def ocr_page(pix, lang='por'):
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    text = pytesseract.image_to_string(img, lang=lang)
    return text

In [ ]:
import json
from pathlib import Path


CACHE_DIR = Path("../../data/cache/ocr")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def cache_key(document_id, page_num, dpi):
    return CACHE_DIR / f"{document_id}_p{page_num}_dpi{dpi}.json"

def get_or_ocr(doc_path, document_id, page_num, dpi=300, lang='por'):
    key = cache_key(document_id, page_num, dpi)
    if key.exists():
        return json.loads(key.read_text())["text"]
    
    pix = rasterize_page(doc_path, page_num, dpi=dpi)

    text = ocr_page(pix, lang=lang)
    key.write_text(
        json.dumps(
            {
                "doc_path": str(doc_path),
                "id": document_id,
                "text": text, 
                "dpi": dpi, 
                "page": page_num
            }
        )
    )
    return text

In [ ]:
get_or_ocr(
    doc_path=doc_path,
    document_id="15414618005202301",
    page_num=3,
    dpi=150,
)

All is working fine. Now, I will test a random set of pages from the PDF, 20% of the total pages, to compare how DPI affects the OCR results.

In [ ]:
DPI_TEST_VALUES = [50, 100, 150, 200, 300, 400, 500, 600]

total_pages = len(fitz.open(doc_path))
n_of_random_pages = max(1, total_pages // 5)
print(f"total pages in document: {total_pages}")
print(f"number of pages to test that represent 20%: {n_of_random_pages}")

In [ ]:
import random

random_pages = random.sample(range(total_pages), n_of_random_pages)
print(f"Random pages to test: {random_pages}")

In [ ]:
for page_num in random_pages:
    print(f"\n\nPage {page_num}")
    for dpi in DPI_TEST_VALUES:
        text = get_or_ocr(
            doc_path=doc_path,
            document_id="15414618005202301",
            page_num=page_num,
            dpi=dpi,
        )
        print(f"DPI: {dpi}, Text length: {len(text)}")


In the test I made, this was the result of the OCR for the 20% of the pages:

```txt
Page 4
DPI: 50, Text length: 1730
DPI: 100, Text length: 2107
DPI: 150, Text length: 2118
DPI: 200, Text length: 2118
DPI: 300, Text length: 2118
DPI: 400, Text length: 2118
DPI: 500, Text length: 2120
DPI: 600, Text length: 2120


Page 26
DPI: 50, Text length: 268
DPI: 100, Text length: 987
DPI: 150, Text length: 1025
DPI: 200, Text length: 1010
DPI: 300, Text length: 1008
DPI: 400, Text length: 1006
DPI: 500, Text length: 1010
DPI: 600, Text length: 1024


Page 0
DPI: 50, Text length: 91
DPI: 100, Text length: 99
DPI: 150, Text length: 99
DPI: 200, Text length: 106
DPI: 300, Text length: 101
DPI: 400, Text length: 100
DPI: 500, Text length: 100
DPI: 600, Text length: 100


Page 9
DPI: 50, Text length: 2332
DPI: 100, Text length: 2568
DPI: 150, Text length: 2571
DPI: 200, Text length: 2571
DPI: 300, Text length: 2571
DPI: 400, Text length: 2571
DPI: 500, Text length: 2570
DPI: 600, Text length: 2569


Page 15
DPI: 50, Text length: 2401
DPI: 100, Text length: 2575
DPI: 150, Text length: 2578
DPI: 200, Text length: 2578
DPI: 300, Text length: 2575
DPI: 400, Text length: 2576
DPI: 500, Text length: 2574
DPI: 600, Text length: 2575
```


I decided to use the DPI of 150 for the OCR processing because it is the lowest DPI that gives a good result, based on the text length. The text length is not a perfect metric, but it is a cheap way to compare the results.

### Conclusion and validation

The DPI of 150 is a good compromise between performance and quality. Let's do a spot check of the OCR results to validate the quality of the text extracted. A special attention should be given to the OCR results of the pages that have numbers of clauses, because any mistake in the OCR of these numbers can lead to a wrong interpretation of the document and propagate errors in the pipeline.

In [ ]:
from datetime import datetime
import random
import fitz
from pathlib import Path

doc_path = Path("../../data/policies/raw/15414618005202301.pdf")

total_pages = len(fitz.open(doc_path))
n_of_random_pages = 10

random_pages = random.sample(range(total_pages), n_of_random_pages)

for page_num in random_pages:
    for dpi in [150]:
        text = get_or_ocr(
            doc_path=doc_path,
            document_id="15414618005202301",
            page_num=page_num,
            dpi=dpi,
        )
        print(f"\n\nText extracted for page {page_num + 1} at DPI {dpi}:\n{text}")
            

In [ ]:
random_pages

I analyzed this random pages: [25, 3, 24, 1, 7, 8, 16, 13, 21, 23]

I found only one issue. The list of chapters on page 3 isn't extracting perfectly. Let's check it out:

```bash
Text extracted for page 3 at DPI 150:
DNS na Wan

(LMG) E REINTEGRAÇÃO

9.

10.
11.
12.
13.
14.
15.
16.
17.
18.
19.
20.
21.
22.
23.

too

seguros

ÍNDICE

OBJETIVO DO SEGURO...
COBERTURAS DO SEGURO.. nn
RISCOS EXCLUÍDOS................. ri reeeeeeeeeeremeeererereenees 4
BENS NÃO COBERTOS NO SEGURO
FORMA DE CONTRATAÇÃO DO SEGURO
ACEITAÇÃO, CONTRATAÇÃO E VIGÊNCIA .................c cities 9
RENOVAÇÃO................ce iii creeeeeerererererereeaeaeeearerararaeererereranes n
LIMITE MÁXIMO DE INDENIZAÇÃO (LMI), LIMITE MÁXIMO DE GARANTIA

CARÊNCIA E FRANQUIA...
PAGAMENTO DE PRÊMIO E REAJUSTE DO PRÊMIO ....................i. 11
ÂMBITO GEOGRÁFICO DA COBERTURA
COMUNICAÇÃO E DOCUMENTOS DE SINISTRO .
INDENIZAÇÃO DE SINISTROS.................ci rs rtrremeteeeeeeeeeeereerereerereess
ATUALIZAÇÃO DAS OBRIGAÇÕES CONTRATUAIS.
CONCORRÊNCIA DE SEGUROS..
PERDA DO DIREITO À INDENIZAÇÃO ..
CANCELAMENTO E RESCISÃO CONTRATUAL
AUDITORIA.
SUB-ROGAÇÃO.

DISPOSIÇÕES FINAIS a
GLOSSÁRIO.............. ir eirerrerreceeaeeree aee reerre aerea cre reereerrertertesa

Processo SUSEP Nº
(Condições Gerais Seguro Assistência e Outras Coberturas - Auto) 3
```

Despite this, I'm not considering it a problem because the most critical information, present on the other pages, extracted correctly.

So, we can use a DPI of 150. Therefore, the manual analysis of 10 random pages validates my qualitative intuition about the method applied.

But, I'd like to see the other document that also need a OCR approach.

In [ ]:
import fitz
import random

doc_path_kovr = Path("../../data/policies/raw/15414604545202481.pdf")
document_id_kovr = "15414604545202481"

DPI_TEST_VALUES = [50, 100, 150, 200, 300]

total_pages_kovr = len(fitz.open(doc_path_kovr))
n_of_random_pages_kovr = max(1, total_pages_kovr // 5)
print(f"total pages in document: {total_pages_kovr}")
print(f"number of pages to test (20%): {n_of_random_pages_kovr}")

random.seed(42)
random_pages_kovr = random.sample(range(total_pages_kovr), n_of_random_pages_kovr)
print(f"Random pages to test: {random_pages_kovr}")

for page_num in random_pages_kovr:
    print(f"\n\nPage {page_num}")
    for dpi in DPI_TEST_VALUES:
        text = get_or_ocr(
            doc_path=doc_path_kovr,
            document_id=document_id_kovr,
            page_num=page_num,
            dpi=dpi,
        )
        print(f"DPI: {dpi}, Text length: {len(text)}")

The result was:

```bash
total pages in document: 33
number of pages to test (20%): 6
Random pages to test: [7, 1, 23, 8, 32, 28]


Page 7
DPI: 50, Text length: 2029
DPI: 100, Text length: 3842
DPI: 150, Text length: 3859
DPI: 200, Text length: 3874
DPI: 300, Text length: 3873


Page 1
DPI: 50, Text length: 604
DPI: 100, Text length: 1071
DPI: 150, Text length: 1080
DPI: 200, Text length: 1079
DPI: 300, Text length: 1078


Page 23
DPI: 50, Text length: 1441
DPI: 100, Text length: 3477
DPI: 150, Text length: 3487
DPI: 200, Text length: 3505
DPI: 300, Text length: 3494


Page 8
DPI: 50, Text length: 2301
DPI: 100, Text length: 3804
DPI: 150, Text length: 3822
DPI: 200, Text length: 3840
DPI: 300, Text length: 3837


Page 32
DPI: 50, Text length: 73
DPI: 100, Text length: 560
DPI: 150, Text length: 576
DPI: 200, Text length: 574
DPI: 300, Text length: 574


Page 28
DPI: 50, Text length: 2034
DPI: 100, Text length: 3607
DPI: 150, Text length: 3620
DPI: 200, Text length: 3623
DPI: 300, Text length: 3620
```

In [ ]:
random.seed(7)
spot_check_pages_kovr = random.sample(range(total_pages_kovr), 5)

for page_num in spot_check_pages_kovr:
    text = get_or_ocr(
        doc_path=doc_path_kovr,
        document_id=document_id_kovr,
        page_num=page_num,
        dpi=150,
    )
    print(f"\n\nText extracted for page {page_num + 1} at DPI 150:\n{text}")

Both target documents converge on DPI 150 as the point where OCR output stabilizes, and clause numbering held up correctly across every sampled page in both cases despite unrelated OCR noise (header/logo garbling, a TOC page, one visibly weaker page in the KOVR document), validating DPI 150 as the setting for the M1-02 OCR path.